
### SILVER LAYER SCRIPT

=========================================
### SILVER LAYER - ORDER ITEMS
###Source  : Bronze (ADLS)
### Target  : Delta Silver Table
### Load    : Incremental Append
 =========================================

In [0]:
df_bronze_order_items = spark.read.csv(
    "abfss://bronze@incdata.dfs.core.windows.net/olist/order_items",
    header=True
)

display(df_bronze_order_items)


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23 03:55:27,21.90,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14 12:10:31,19.90,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10 12:30:45,810.00,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26 18:31:29,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06 14:10:56,53.99,11.40


### CAST DATA TYPES

In [0]:
from pyspark.sql.functions import col, to_timestamp

df_silver_order_items = (
    df_bronze_order_items
    .withColumn("order_item_id", col("order_item_id").cast("int"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("freight_value", col("freight_value").cast("double"))
    .withColumn(
        "shipping_limit_date",
        to_timestamp("shipping_limit_date")
    )
)


### REMOVE NULLS

In [0]:
df_silver_order_items = df_silver_order_items.filter(
    col("price").isNotNull() &
    col("freight_value").isNotNull()
)


In [0]:
df_silver_order_items.printSchema()
display(df_silver_order_items)


root
 |-- order_id: string (nullable = true)
 |-- order_item_id: integer (nullable = true)
 |-- product_id: string (nullable = true)
 |-- seller_id: string (nullable = true)
 |-- shipping_limit_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- freight_value: double (nullable = true)



order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19T09:45:35Z,58.9,13.29
00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03T11:05:13Z,239.9,19.93
000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18T14:48:30Z,199.0,17.87
00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15T10:10:18Z,12.99,12.79
00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13T13:57:51Z,199.9,18.14
00048cc3ae777c65dbb7d2a0634bc1ea,1,ef92defde845ab8450f9d70c526ef70f,6426d21aca402a131fc0a5d0960a3c90,2017-05-23T03:55:27Z,21.9,12.69
00054e8431b9d7675808bcb819fb4a32,1,8d4f2bb7e93e6710a28f34fa83ee7d28,7040e82f899a04d1b434b795a43b4617,2017-12-14T12:10:31Z,19.9,11.85
000576fe39319847cbb9d288c5617fa6,1,557d850972a7d6f792fd18ae1400d9b6,5996cddab893a4652a15592fb58ab8db,2018-07-10T12:30:45Z,810.0,70.75
0005a1a1728c9d785b8e2b08b904576c,1,310ae3c140ff94b03219ad0adc3c778f,a416b6a846a11724393025641d4edd5e,2018-03-26T18:31:29Z,145.95,11.65
0005f50442cb953dcd1d21e1fb923495,1,4535b0e1091c278dfd193e5a1d63b39f,ba143b05f0110f0dc71ad71b4466ce92,2018-07-06T14:10:56Z,53.99,11.4


### WRITE TO SILVER

In [0]:
(
    df_silver_order_items
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("olist_silver_order_items")
)


In [0]:
%sql
SELECT COUNT(*) FROM olist_silver_order_items;


count(1)
112650


In [0]:
%sql
SELECT * 
FROM olist_silver_order_items 
LIMIT 10;


order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
cc82a53fe1a3af9541b2ae5c8290262a,1,4af81c9413dcb40e03e2a1bc15a6448b,ea8482cd71df3c1969d7b9473ff13abc,2017-09-01T15:30:22Z,48.9,7.78
cc82b75db7d018c137b470d668b91996,1,6c712952b8ef62f8d06a0314917400c8,620c87c171fb2a6dd6e8bb4dec959fc6,2017-12-27T19:32:20Z,75.9,12.49
cc8304651e8857627e56aca5207db748,1,a2da86fa759178e9e58e54aa1a144e59,ea8482cd71df3c1969d7b9473ff13abc,2018-02-14T07:49:37Z,24.99,15.1
cc8389c87998ef48d592285b08d5106d,1,edcda2e50aadade37544f55a8da52207,ea8482cd71df3c1969d7b9473ff13abc,2018-03-08T22:49:27Z,24.99,14.1
cc8412c7495b3d013e638dae26f04c9a,1,d2f5484cbffe4ca766301b21ab9246dd,36a968b544695394e4e9d7572688598f,2017-08-28T12:26:32Z,12.88,8.27
cc8454f08599794be25a95f691617e11,1,26b2a480ced2900a62bb9431133e95aa,d6b1ce66b035a475f00c017792ff9769,2018-06-29T20:10:19Z,113.3,15.1
cc84d04f45b429e9b55417f15b01aa52,1,ee0c1cf2fbeae95205b4aa506f1469f0,cc419e0650a3c5ba77189a1882b7556a,2018-03-30T05:27:44Z,53.99,12.82
cc852cf36cf64a124456f5f10b2142ae,1,4df83a41105e00e0845b9d12b5fe601c,fa1c13f2614d7b5c4749cbc52fecda94,2018-01-17T15:28:32Z,858.9,13.27
cc85879fb8f1767ad9b67dbcfa113e7b,1,d3e1006ba3735c0d44160026b6e0ced3,c003204e1ab016dfa150abc119207b24,2018-04-13T19:29:41Z,109.9,21.85
cc85b0be56a5a15d2146816d54ee0eff,1,2ffdf10e724b958c0f7ea69e97d32f64,4869f7a5dfa277a7dca6462dcf3b52b2,2018-01-23T03:38:42Z,213.9,16.25


### INCREMENTAL LOGIC

In [0]:
(
    df_silver_order_items
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("olist_silver_order_items")
)


In [0]:
%sql SHOW TABLES;


database,tableName,isTemporary
default,olist_silver_customers,false
default,olist_silver_order_items,false
default,olist_silver_orders,false
,_sqldf,true


In [0]:
%sql DESCRIBE olist_silver_order_items


col_name,data_type,comment
order_id,string,null
order_item_id,int,null
product_id,string,null
seller_id,string,null
shipping_limit_date,timestamp,null
price,double,null
freight_value,double,null
